# 📰 NewsIR — News Article Information Retrieval System

A complete IR pipeline that **collects**, **preprocesses**, **indexes**, **searches**, and **evaluates** real-world news articles using BM25 and TF-IDF ranking.

**Pipeline Overview:**
1. Data Collection (RSS feeds + web scraping)
2. Text Preprocessing (tokenization, stemming, lemmatization, stop words)
3. Inverted Index Construction (TF-IDF & BM25)
4. Search & Retrieval (ranked results with keyword highlighting)
5. Evaluation (Precision, Recall, F1, MAP, NDCG)

---

**Technologies:** Python, NLTK, BeautifulSoup, Feedparser, Flask

**Data Sources:** BBC News, Reuters, CNN, The Guardian, NPR, Al Jazeera

## 1. Setup & Dependencies

First, we install and import all required libraries for the IR system.

In [ ]:
!pip install nltk requests beautifulsoup4 feedparser flask lxml_html_clean -q

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print('NLTK data ready.')

### 📘 Explanation — Setup & Dependencies

The system relies on several key libraries:

| Library | Purpose |
|---------|---------|
| **NLTK** | Natural Language Toolkit — provides tokenization (`word_tokenize`), stop words list, Porter Stemmer, and WordNet Lemmatizer |
| **requests** | HTTP library used to fetch full article content from news websites |
| **BeautifulSoup** | HTML parser that extracts readable text from raw web pages |
| **feedparser** | Parses RSS/Atom feeds to collect article metadata (title, link, summary) |
| **Flask** | Lightweight web framework powering the search UI |
| **lxml_html_clean** | HTML sanitizer used internally by feedparser/BS4 |

**NLTK Data Downloads:**
- `punkt` / `punkt_tab` — sentence and word tokenizer models
- `stopwords` — list of common English words to filter out (e.g., "the", "is", "at")
- `wordnet` — lexical database used by the WordNet Lemmatizer to find base forms of words (e.g., "running" → "run")

---
## 2. Data Collection

We collect news articles from **10 real-world RSS feeds** (BBC, CNN, NPR, The Guardian, Reuters) using `feedparser` and extract full article text with `BeautifulSoup`.

In [ ]:
"""
Data Collection Module for News IR System
==========================================
Collects news articles from real-world sources using RSS feeds and web scraping.
Sources: BBC News, Reuters, CNN, Al Jazeera, The Guardian, NPR
"""

import os
import json
import time
import hashlib
import requests
import feedparser
from datetime import datetime
from bs4 import BeautifulSoup

# RSS Feed sources for news articles
RSS_FEEDS = {
    "BBC News": "http://feeds.bbci.co.uk/news/rss.xml",
    "BBC Technology": "http://feeds.bbci.co.uk/news/technology/rss.xml",
    "BBC Science": "http://feeds.bbci.co.uk/news/science_and_environment/rss.xml",
    "Reuters Top News": "https://feeds.reuters.com/reuters/topNews",
    "Reuters Technology": "https://feeds.reuters.com/reuters/technologyNews",
    "NPR News": "https://feeds.npr.org/1001/rss.xml",
    "Al Jazeera": "https://www.aljazeera.com/xml/rss/all.xml",
    "The Guardian World": "https://www.theguardian.com/world/rss",
    "The Guardian Tech": "https://www.theguardian.com/technology/rss",
    "CNN Top Stories": "http://rss.cnn.com/rss/edition.rss",
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}


def generate_doc_id(url):
    """Generate a unique document ID from URL."""
    return hashlib.md5(url.encode()).hexdigest()[:12]


def fetch_article_content(url, timeout=10):
    """Fetch and extract the main text content from a news article URL."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=timeout)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unwanted elements
        for tag in soup.find_all(["script", "style", "nav", "footer", "header", "aside", "iframe"]):
            tag.decompose()

        # Try to find article body using common selectors
        article_body = None
        selectors = [
            "article",
            '[role="main"]',
            ".article-body",
            ".story-body",
            ".article__body",
            ".post-content",
            ".entry-content",
            "#article-body",
            ".content__article-body",
        ]
        for selector in selectors:
            article_body = soup.select_one(selector)
            if article_body:
                break

        if not article_body:
            article_body = soup.find("body")

        if article_body:
            paragraphs = article_body.find_all("p")
            text = " ".join(p.get_text(strip=True) for p in paragraphs if len(p.get_text(strip=True)) > 30)
            return text if len(text) > 100 else None
        return None

    except Exception as e:
        print(f"  [WARN] Could not fetch {url}: {e}")
        return None


def collect_from_rss(max_per_feed=15, fetch_full_text=True):
    """
    Collect news articles from RSS feeds.
    
    Args:
        max_per_feed: Maximum articles to collect per feed
        fetch_full_text: Whether to fetch full article text from URLs
        
    Returns:
        List of article dictionaries
    """
    articles = []
    seen_urls = set()

    print("=" * 60)
    print("  NEWS DATA COLLECTION")
    print("=" * 60)

    for source_name, feed_url in RSS_FEEDS.items():
        print(f"\n[*] Fetching from: {source_name}")
        try:
            feed = feedparser.parse(feed_url)
            entries = feed.entries[:max_per_feed]
            count = 0

            for entry in entries:
                url = entry.get("link", "")
                if not url or url in seen_urls:
                    continue
                seen_urls.add(url)

                title = entry.get("title", "").strip()
                summary = entry.get("summary", entry.get("description", "")).strip()

                # Clean HTML from summary
                if summary:
                    summary = BeautifulSoup(summary, "html.parser").get_text(strip=True)

                published = entry.get("published", entry.get("updated", ""))

                # Fetch full article text
                full_text = None
                if fetch_full_text and url:
                    full_text = fetch_article_content(url)
                    time.sleep(0.5)  # Be polite to servers

                # Use full text if available, otherwise fall back to summary
                content = full_text if full_text else summary

                if not title or not content or len(content) < 50:
                    continue

                doc_id = generate_doc_id(url)
                article = {
                    "doc_id": doc_id,
                    "title": title,
                    "content": content,
                    "summary": summary[:500] if summary else "",
                    "source": source_name,
                    "url": url,
                    "published_date": published,
                    "collected_at": datetime.now().isoformat(),
                    "has_full_text": full_text is not None,
                }
                articles.append(article)
                count += 1
                print(f"  [{count}] {title[:60]}...")

            print(f"  -> Collected {count} articles from {source_name}")

        except Exception as e:
            print(f"  [ERROR] Failed to fetch {source_name}: {e}")

    print(f"\n{'=' * 60}")
    print(f"  TOTAL ARTICLES COLLECTED: {len(articles)}")
    print(f"{'=' * 60}")
    return articles


def save_articles(articles, filepath="data/raw_articles.json"):
    """Save collected articles to JSON file."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(articles, f, indent=2, ensure_ascii=False)
    print(f"[+] Saved {len(articles)} articles to {filepath}")
    return filepath


def load_articles(filepath="data/raw_articles.json"):
    """Load articles from JSON file."""
    if not os.path.exists(filepath):
        print(f"[!] File not found: {filepath}")
        return []
    with open(filepath, "r", encoding="utf-8") as f:
        articles = json.load(f)
    print(f"[+] Loaded {len(articles)} articles from {filepath}")
    return articles



### 📘 Explanation — Data Collection Module

This module handles the first stage of the IR pipeline: **acquiring raw data**.

**Key Components:**

1. **RSS Feed Sources (`RSS_FEEDS` dict):**
   - 10 feeds from 6 major news organizations (BBC, Reuters, NPR, Al Jazeera, The Guardian, CNN)
   - Covers general news, technology, science, and world topics for diverse vocabulary

2. **`generate_doc_id(url)`:**
   - Creates a unique 12-character document ID using MD5 hash of the URL
   - Ensures each article has a consistent, reproducible identifier

3. **`fetch_article_content(url)`:**
   - Makes an HTTP GET request with a browser-like User-Agent header
   - Removes unwanted HTML elements (scripts, nav, footer, ads)
   - Tries multiple CSS selectors (`article`, `[role="main"]`, `.article-body`, etc.) to find the main content
   - Extracts paragraph text, filtering out short paragraphs (< 30 chars) to skip navigation/boilerplate
   - Returns `None` if content is too short (< 100 chars)

4. **`collect_from_rss(max_per_feed, fetch_full_text)`:**
   - Iterates through all RSS feeds using `feedparser.parse()`
   - Deduplicates articles by URL using a `seen_urls` set
   - Cleans HTML from RSS summaries with BeautifulSoup
   - Optionally fetches full article text (with 0.5s delay to be polite to servers)
   - Falls back to RSS summary if full text fetch fails
   - Skips articles with missing title or content < 50 chars

5. **`save_articles()` / `load_articles()`:**
   - Persist raw articles as JSON for reproducibility
   - Creates the `data/` directory automatically

### Run Data Collection
Collect articles from RSS feeds. Set `fetch_full_text=False` for faster execution.

In [ ]:
articles = collect_from_rss(max_per_feed=10, fetch_full_text=False)
save_articles(articles)
print(f'\nCollected {len(articles)} articles')
print(f'Sources: {set(a["source"] for a in articles)}')

### 📘 Explanation — Running Data Collection

- `max_per_feed=10`: Limits collection to 10 articles per RSS feed (up to ~100 total)
- `fetch_full_text=False`: Uses only RSS summaries instead of scraping full articles (much faster, but shorter content)
- `save_articles()`: Writes collected articles to `data/raw_articles.json`
- Each article dict contains: `doc_id`, `title`, `content`, `summary`, `source`, `url`, `published_date`, `collected_at`, `has_full_text`

### Preview Collected Articles

In [ ]:
import pandas as pd
df_articles = pd.DataFrame(articles)
print(f'Total articles: {len(df_articles)}')
print(f'Columns: {list(df_articles.columns)}')
df_articles[['doc_id', 'title', 'source']].head(10)

### 📘 Explanation — Data Preview

We use pandas DataFrame to inspect the collected dataset:
- **Total articles**: Number of successfully collected news articles
- **Columns**: The fields stored for each article (doc_id, title, content, source, url, etc.)
- The preview shows the first 10 rows with doc_id, title, and source columns to verify data quality and diversity

---
## 3. Text Preprocessing

The preprocessing pipeline applies 5 steps:
1. **Text Normalization** — lowercase, remove URLs/HTML/special chars
2. **Tokenization** — NLTK word_tokenize
3. **Stop Word Removal** — English stopwords + custom news terms
4. **Lemmatization** — WordNet Lemmatizer (dictionary base form)
5. **Stemming** — Porter Stemmer (root form)

In [ ]:
"""
Text Preprocessing Module for News IR System
==============================================
Handles: Tokenization, Stemming, Lemmatization, Stop Word Removal, Text Normalization
"""

import re
import json
import os
from collections import Counter

import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer


class TextPreprocessor:
    """
    Comprehensive text preprocessor for IR system.
    Supports tokenization, normalization, stop word removal,
    stemming, and lemmatization.
    """

    def __init__(self, use_stemming=True, use_lemmatization=True, remove_stopwords=True):
        self.stemmer = PorterStemmer()
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words("english"))
        # Add custom stop words common in news articles
        self.stop_words.update([
            "said", "also", "would", "could", "one", "two", "new",
            "like", "may", "us", "get", "make", "know", "say",
            "reuters", "bbc", "cnn", "guardian", "npr", "ap",
        ])
        self.use_stemming = use_stemming
        self.use_lemmatization = use_lemmatization
        self.remove_stopwords = remove_stopwords

        # Preprocessing statistics
        self.stats = {
            "total_docs_processed": 0,
            "total_tokens_before": 0,
            "total_tokens_after": 0,
            "avg_doc_length_before": 0,
            "avg_doc_length_after": 0,
        }

    def normalize_text(self, text):
        """
        Text normalization:
        - Convert to lowercase
        - Remove URLs, emails, HTML tags
        - Remove special characters and numbers
        - Normalize whitespace
        """
        if not text:
            return ""

        # Lowercase
        text = text.lower()

        # Remove URLs
        text = re.sub(r"https?://\S+|www\.\S+", "", text)

        # Remove email addresses
        text = re.sub(r"\S+@\S+", "", text)

        # Remove HTML tags
        text = re.sub(r"<[^>]+>", "", text)

        # Remove special characters but keep apostrophes in contractions
        text = re.sub(r"[^a-zA-Z\s']", " ", text)

        # Remove standalone apostrophes
        text = re.sub(r"\s'|'\s", " ", text)

        # Normalize whitespace
        text = re.sub(r"\s+", " ", text).strip()

        return text

    def tokenize(self, text):
        """Tokenize text into words using NLTK word tokenizer."""
        try:
            tokens = word_tokenize(text)
        except Exception:
            tokens = text.split()
        # Filter very short tokens
        return [t for t in tokens if len(t) > 1]

    def remove_stop_words(self, tokens):
        """Remove stop words from token list."""
        return [t for t in tokens if t not in self.stop_words]

    def stem_tokens(self, tokens):
        """Apply Porter stemming to tokens."""
        return [self.stemmer.stem(t) for t in tokens]

    def lemmatize_tokens(self, tokens):
        """Apply WordNet lemmatization to tokens."""
        return [self.lemmatizer.lemmatize(t) for t in tokens]

    def preprocess(self, text):
        """
        Full preprocessing pipeline:
        1. Text Normalization
        2. Tokenization
        3. Stop Word Removal
        4. Lemmatization (reduces to dictionary form)
        5. Stemming (reduces to root form)

        NOTE: We apply lemmatization first, then stemming.
        Stemming is the final step so the index uses stemmed forms.

        Returns processed token list.
        """
        # Step 1: Normalize
        normalized = self.normalize_text(text)

        # Step 2: Tokenize
        tokens = self.tokenize(normalized)
        tokens_before = len(tokens)

        # Step 3: Remove stop words
        if self.remove_stopwords:
            tokens = self.remove_stop_words(tokens)

        # Step 4: Lemmatize
        if self.use_lemmatization:
            tokens = self.lemmatize_tokens(tokens)

        # Step 5: Stem
        if self.use_stemming:
            tokens = self.stem_tokens(tokens)

        # Update stats
        self.stats["total_tokens_before"] += tokens_before
        self.stats["total_tokens_after"] += len(tokens)

        return tokens

    def preprocess_keep_original(self, text):
        """
        Preprocess text but also return a mapping from stems back to
        original normalized words. Used for snippet highlighting.
        
        Returns:
            (tokens, stem_to_original_map)
        """
        normalized = self.normalize_text(text)
        tokens = self.tokenize(normalized)
        
        if self.remove_stopwords:
            tokens = self.remove_stop_words(tokens)
        
        original_tokens = list(tokens)  # Save before stemming/lemmatization
        
        if self.use_lemmatization:
            tokens = self.lemmatize_tokens(tokens)
        if self.use_stemming:
            tokens = self.stem_tokens(tokens)
        
        # Build reverse map: stemmed -> set of originals
        stem_to_original = {}
        for orig, stemmed in zip(original_tokens, tokens):
            if stemmed not in stem_to_original:
                stem_to_original[stemmed] = set()
            stem_to_original[stemmed].add(orig)
        
        return tokens, stem_to_original

    def get_original_words_for_query(self, query):
        """
        For a query string, return both the processed tokens AND the
        original normalized words (for matching in snippets).
        """
        normalized = self.normalize_text(query)
        original_words = self.tokenize(normalized)
        if self.remove_stopwords:
            original_words = self.remove_stop_words(original_words)
        
        processed = list(original_words)
        if self.use_lemmatization:
            processed = self.lemmatize_tokens(processed)
        if self.use_stemming:
            processed = self.stem_tokens(processed)
        
        return processed, original_words

    def preprocess_documents(self, articles):
        """
        Preprocess a collection of news articles.

        Args:
            articles: List of article dicts with 'title', 'content', 'doc_id'

        Returns:
            List of processed document dicts
        """
        processed_docs = []
        print("\n" + "=" * 60)
        print("  TEXT PREPROCESSING")
        print("=" * 60)

        for i, article in enumerate(articles):
            doc_id = article["doc_id"]
            title = article.get("title", "")
            content = article.get("content", "")

            # Preprocess title and content separately
            title_tokens = self.preprocess(title)
            content_tokens = self.preprocess(content)

            # Title boosting: add title tokens with higher weight (3x)
            tokens = title_tokens * 3 + content_tokens

            if len(tokens) < 5:
                continue

            processed_doc = {
                "doc_id": doc_id,
                "title": title,
                "original_content": content[:2000],  # Store more content for better snippets
                "tokens": tokens,
                "token_count": len(tokens),
                "source": article.get("source", ""),
                "url": article.get("url", ""),
                "published_date": article.get("published_date", ""),
            }
            processed_docs.append(processed_doc)

            if (i + 1) % 20 == 0 or i == 0:
                print(f"  Processed {i + 1}/{len(articles)} documents...")

        self.stats["total_docs_processed"] = len(processed_docs)
        if processed_docs:
            self.stats["avg_doc_length_before"] = round(
                self.stats["total_tokens_before"] / len(processed_docs), 1
            )
            self.stats["avg_doc_length_after"] = round(
                self.stats["total_tokens_after"] / len(processed_docs), 1
            )

        print(f"\n  Preprocessing Statistics:")
        print(f"  - Documents processed: {self.stats['total_docs_processed']}")
        print(f"  - Avg tokens before:   {self.stats['avg_doc_length_before']}")
        print(f"  - Avg tokens after:    {self.stats['avg_doc_length_after']}")
        reduction = 0
        if self.stats['total_tokens_before'] > 0:
            reduction = (1 - self.stats['total_tokens_after'] / self.stats['total_tokens_before']) * 100
        print(f"  - Token reduction:     {reduction:.1f}%")
        print("=" * 60)

        return processed_docs

    def get_vocabulary_stats(self, processed_docs):
        """Get vocabulary statistics from processed documents."""
        all_tokens = []
        for doc in processed_docs:
            all_tokens.extend(doc["tokens"])

        vocab = set(all_tokens)
        freq = Counter(all_tokens)
        most_common = freq.most_common(30)

        return {
            "vocabulary_size": len(vocab),
            "total_tokens": len(all_tokens),
            "most_common_30": most_common,
            "avg_frequency": round(len(all_tokens) / max(len(vocab), 1), 2),
        }


def save_processed(processed_docs, filepath="data/processed_articles.json"):
    """Save processed documents."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(processed_docs, f, indent=2, ensure_ascii=False)
    print(f"[+] Saved {len(processed_docs)} processed documents to {filepath}")


def load_processed(filepath="data/processed_articles.json"):
    """Load processed documents."""
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)


### 📘 Explanation — Text Preprocessing Module

This is the most critical module for IR quality. It transforms raw text into normalized tokens that the index can efficiently store and match.

**Class: `TextPreprocessor`**

**`__init__` parameters:**
- `use_stemming=True`: Enable Porter Stemming (e.g., "running" → "run")
- `use_lemmatization=True`: Enable WordNet Lemmatization (e.g., "better" → "good")
- `remove_stopwords=True`: Filter out common words

**Custom Stop Words:** Adds news-specific terms like "said", "reuters", "bbc", "cnn" that appear frequently but carry no search value.

**Pipeline Methods (applied in order):**

| Step | Method | What it does | Example |
|------|--------|-------------|---------|
| 1 | `normalize_text()` | Lowercase, remove URLs/emails/HTML, remove special chars, normalize whitespace | "AI is GREAT!" → "ai is great" |
| 2 | `tokenize()` | NLTK `word_tokenize`, filter tokens < 2 chars | "ai is great" → ["ai", "is", "great"] |
| 3 | `remove_stop_words()` | Remove English stopwords + custom news terms | ["ai", "is", "great"] → ["ai", "great"] |
| 4 | `lemmatize_tokens()` | WordNet Lemmatizer — reduces to dictionary form | ["countries", "running"] → ["country", "running"] |
| 5 | `stem_tokens()` | Porter Stemmer — reduces to root form | ["country", "running"] → ["countri", "run"] |

**Why Lemmatization THEN Stemming?**
- Lemmatization first produces valid dictionary words
- Stemming then aggressively reduces to root forms for better recall
- The index stores stemmed forms, so "technology", "technologies", "technological" all map to "technolog"

**`preprocess_documents()`:**
- Processes each article's title and content separately
- **Title boosting**: Title tokens are repeated 3× to give them higher weight in the index
- Skips documents with < 5 tokens after preprocessing
- Tracks statistics: tokens before/after, reduction percentage

**`preprocess_keep_original()`:** Returns both stemmed tokens AND a reverse map (stem → original words), used for snippet highlighting.

**`get_vocabulary_stats()`:** Computes vocabulary size, total token count, term frequencies, and top-30 most common terms.

### Run Preprocessing

In [ ]:
from data_collector import load_articles
articles = load_articles('data/raw_articles.json')

preprocessor = TextPreprocessor(use_stemming=True, use_lemmatization=True)
processed_docs = preprocessor.preprocess_documents(articles)
save_processed(processed_docs)

### 📘 Explanation — Running Preprocessing

1. **Load raw articles** from `data/raw_articles.json`
2. **Initialize `TextPreprocessor`** with both stemming and lemmatization enabled
3. **`preprocess_documents()`** applies the full 5-step pipeline to every article:
   - Each article's title is preprocessed and repeated 3× (title boosting)
   - Content tokens are appended
   - Documents with < 5 final tokens are discarded
4. **`save_processed()`** writes processed documents to `data/processed_articles.json`

The output shows processing progress and statistics about token reduction.

### Preprocessing Example
Let's see how a sample text gets transformed through each step:

In [ ]:
sample = "The United States government announced new Artificial Intelligence regulations for technology companies."
p = TextPreprocessor()

print("Original:", sample)
print("Normalized:", p.normalize_text(sample))

tokens = p.tokenize(p.normalize_text(sample))
print("Tokenized:", tokens)

no_stop = p.remove_stop_words(tokens)
print("Stop words removed:", no_stop)

lemmatized = p.lemmatize_tokens(no_stop)
print("Lemmatized:", lemmatized)

stemmed = p.stem_tokens(lemmatized)
print("Stemmed:", stemmed)

print("\nFinal pipeline output:", p.preprocess(sample))

### 📘 Explanation — Preprocessing Step-by-Step Demo

This cell demonstrates each preprocessing step on a sample sentence:

1. **Original**: "The United States government announced new Artificial Intelligence regulations for technology companies."
2. **Normalized**: Converts to lowercase, removes special characters
3. **Tokenized**: Splits into individual word tokens using NLTK
4. **Stop words removed**: Filters out "the", "new", "for" etc.
5. **Lemmatized**: "regulations" → "regulation", "companies" → "company"
6. **Stemmed**: "regulation" → "regul", "intelligence" → "intellig", "technology" → "technolog"
7. **Final output**: The complete pipeline result — this is what gets stored in the index

### Vocabulary Statistics

In [ ]:
vocab_stats = preprocessor.get_vocabulary_stats(processed_docs)
print(f"Vocabulary size: {vocab_stats['vocabulary_size']}")
print(f"Total tokens: {vocab_stats['total_tokens']}")
print(f"Avg frequency: {vocab_stats['avg_frequency']}")
print(f"\nTop 20 most frequent terms:")
for term, freq in vocab_stats['most_common_30'][:20]:
    print(f"  {term:20s} -> {freq}")

### 📘 Explanation — Vocabulary Statistics

- **Vocabulary size**: Number of unique terms after preprocessing (typically 4,000–7,000 for ~80 articles)
- **Total tokens**: Sum of all token occurrences across all documents
- **Avg frequency**: Average number of times each term appears (total_tokens / vocabulary_size)
- **Top 20 terms**: Most frequent terms in the corpus — useful for understanding what topics dominate the dataset
- High-frequency terms that aren't meaningful might suggest additional stop words to add

---
## 4. Inverted Index Construction

We build an **Inverted Index** mapping each term to its posting list:
```
index[term] = {doc_id_1: freq_1, doc_id_2: freq_2, ...}
```

**Scoring functions:**
- **TF-IDF**: `TF(log-normalized) × IDF`
- **BM25 (Okapi)**: `Σ IDF(t) × [tf × (k1+1)] / [tf + k1 × (1-b+b×|d|/avgdl)]` with k1=1.5, b=0.75

In [ ]:
"""
Indexing Module for News IR System
====================================
Builds and manages the Inverted Index for efficient document retrieval.
Supports TF-IDF weighting and BM25 scoring.
"""

import os
import json
import math
import pickle
from collections import defaultdict, Counter


class InvertedIndex:
    """
    Inverted Index with TF-IDF and BM25 scoring.
    
    Structure:
        index[term] = {doc_id: term_frequency, ...}
    """

    def __init__(self):
        # Core index: term -> {doc_id: tf, ...}
        self.index = defaultdict(dict)
        # Document metadata
        self.documents = {}  # doc_id -> {title, source, url, ...}
        self.doc_lengths = {}  # doc_id -> number of tokens
        self.doc_tokens = {}  # doc_id -> token list (for snippets)
        # Corpus statistics
        self.total_docs = 0
        self.avg_doc_length = 0
        self.vocabulary = set()
        # IDF cache
        self._idf_cache = {}

    def build_index(self, processed_docs):
        """
        Build the inverted index from processed documents.
        
        Args:
            processed_docs: List of dicts with 'doc_id', 'tokens', 'title', etc.
        """
        print("\n" + "=" * 60)
        print("  BUILDING INVERTED INDEX")
        print("=" * 60)

        self.index.clear()
        self.documents.clear()
        self.doc_lengths.clear()

        total_length = 0

        for doc in processed_docs:
            doc_id = doc["doc_id"]
            tokens = doc["tokens"]

            # Store document metadata
            self.documents[doc_id] = {
                "title": doc.get("title", ""),
                "source": doc.get("source", ""),
                "url": doc.get("url", ""),
                "original_content": doc.get("original_content", ""),
                "published_date": doc.get("published_date", ""),
                "token_count": len(tokens),
            }

            self.doc_lengths[doc_id] = len(tokens)
            self.doc_tokens[doc_id] = tokens
            total_length += len(tokens)

            # Count term frequencies
            term_freq = Counter(tokens)

            # Add to inverted index
            for term, freq in term_freq.items():
                self.index[term][doc_id] = freq
                self.vocabulary.add(term)

        self.total_docs = len(self.documents)
        self.avg_doc_length = total_length / max(self.total_docs, 1)

        # Precompute IDF values
        self._compute_idf()

        print(f"  - Documents indexed:   {self.total_docs}")
        print(f"  - Vocabulary size:     {len(self.vocabulary)}")
        print(f"  - Avg document length: {self.avg_doc_length:.1f} tokens")
        print(f"  - Total postings:      {sum(len(v) for v in self.index.values())}")
        print("=" * 60)

    def _compute_idf(self):
        """Precompute IDF for all terms."""
        self._idf_cache = {}
        for term in self.index:
            df = len(self.index[term])
            self._idf_cache[term] = math.log((self.total_docs - df + 0.5) / (df + 0.5) + 1)

    def get_idf(self, term):
        """Get IDF value for a term."""
        return self._idf_cache.get(term, 0)

    def get_tf(self, term, doc_id):
        """Get term frequency in a document."""
        return self.index.get(term, {}).get(doc_id, 0)

    def get_tfidf(self, term, doc_id):
        """Compute TF-IDF score for a term in a document."""
        tf = self.get_tf(term, doc_id)
        if tf == 0:
            return 0
        # Log-normalized TF
        tf_norm = 1 + math.log(tf) if tf > 0 else 0
        idf = self.get_idf(term)
        return tf_norm * idf

    def bm25_score(self, query_terms, doc_id, k1=1.5, b=0.75):
        """
        Compute BM25 score for a document given query terms.
        
        Args:
            query_terms: List of preprocessed query terms
            doc_id: Document ID
            k1: Term frequency saturation parameter
            b: Length normalization parameter
            
        Returns:
            BM25 score
        """
        score = 0
        doc_len = self.doc_lengths.get(doc_id, 0)

        for term in query_terms:
            if term not in self.index or doc_id not in self.index[term]:
                continue

            tf = self.index[term][doc_id]
            idf = self.get_idf(term)

            # BM25 formula
            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * doc_len / self.avg_doc_length)
            score += idf * (numerator / denominator)

        return score

    def search_tfidf(self, query_terms, top_k=10):
        """
        Search using TF-IDF cosine similarity.
        
        Returns:
            List of (doc_id, score) tuples, sorted by score descending
        """
        scores = defaultdict(float)
        
        # Get candidate documents (any doc containing at least one query term)
        candidate_docs = set()
        for term in query_terms:
            if term in self.index:
                candidate_docs.update(self.index[term].keys())

        for doc_id in candidate_docs:
            for term in query_terms:
                scores[doc_id] += self.get_tfidf(term, doc_id)

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

    def search_bm25(self, query_terms, top_k=10):
        """
        Search using BM25 ranking.
        
        Returns:
            List of (doc_id, score) tuples, sorted by score descending
        """
        scores = {}

        # Get candidate documents
        candidate_docs = set()
        for term in query_terms:
            if term in self.index:
                candidate_docs.update(self.index[term].keys())

        for doc_id in candidate_docs:
            scores[doc_id] = self.bm25_score(query_terms, doc_id)

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

    def boolean_search(self, query_terms, mode="AND"):
        """
        Boolean search (AND/OR).
        
        Returns:
            Set of matching document IDs
        """
        if not query_terms:
            return set()

        if mode == "AND":
            result = None
            for term in query_terms:
                docs = set(self.index.get(term, {}).keys())
                result = docs if result is None else result & docs
            return result or set()
        else:  # OR
            result = set()
            for term in query_terms:
                result.update(self.index.get(term, {}).keys())
            return result

    def get_document(self, doc_id):
        """Get document metadata by ID."""
        return self.documents.get(doc_id)

    def get_posting_list(self, term):
        """Get posting list for a term."""
        return self.index.get(term, {})

    def save_index(self, filepath="data/inverted_index.pkl"):
        """Save the inverted index to disk."""
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        data = {
            "index": dict(self.index),
            "documents": self.documents,
            "doc_lengths": self.doc_lengths,
            "doc_tokens": self.doc_tokens,
            "total_docs": self.total_docs,
            "avg_doc_length": self.avg_doc_length,
            "vocabulary": list(self.vocabulary),
            "idf_cache": self._idf_cache,
        }
        with open(filepath, "wb") as f:
            pickle.dump(data, f)
        print(f"[+] Index saved to {filepath}")

    def load_index(self, filepath="data/inverted_index.pkl"):
        """Load the inverted index from disk."""
        if not os.path.exists(filepath):
            print(f"[!] Index file not found: {filepath}")
            return False
        with open(filepath, "rb") as f:
            data = pickle.load(f)
        self.index = defaultdict(dict, data["index"])
        self.documents = data["documents"]
        self.doc_lengths = data["doc_lengths"]
        self.doc_tokens = data.get("doc_tokens", {})
        self.total_docs = data["total_docs"]
        self.avg_doc_length = data["avg_doc_length"]
        self.vocabulary = set(data["vocabulary"])
        self._idf_cache = data["idf_cache"]
        print(f"[+] Index loaded: {self.total_docs} docs, {len(self.vocabulary)} terms")
        return True

    def get_index_stats(self):
        """Return index statistics."""
        return {
            "total_documents": self.total_docs,
            "vocabulary_size": len(self.vocabulary),
            "avg_document_length": round(self.avg_doc_length, 1),
            "total_postings": sum(len(v) for v in self.index.values()),
            "sources": list(set(d.get("source", "") for d in self.documents.values())),
        }


### 📘 Explanation — Inverted Index Module

The inverted index is the core data structure of any IR system. It enables fast lookup of documents containing any given term.

**Class: `InvertedIndex`**

**Data Structures:**
- `self.index`: `defaultdict(dict)` — maps `term → {doc_id: frequency, ...}`
- `self.documents`: Document metadata (title, source, URL, content)
- `self.doc_lengths`: Token count per document (used for BM25 length normalization)
- `self.doc_tokens`: Full token list per document (used for snippet generation)
- `self._idf_cache`: Precomputed IDF values for all terms

**`build_index(processed_docs)`:**
1. For each document, count term frequencies using `Counter(tokens)`
2. For each (term, frequency) pair, add to inverted index: `index[term][doc_id] = freq`
3. Compute corpus statistics: total docs, average document length
4. Precompute IDF values for all terms

**Scoring Functions:**

| Method | Formula | Description |
|--------|---------|-------------|
| `get_idf(term)` | `log((N - df + 0.5) / (df + 0.5) + 1)` | Inverse Document Frequency — rare terms get higher weight |
| `get_tfidf(term, doc_id)` | `(1 + log(tf)) × IDF` | Log-normalized TF × IDF |
| `bm25_score(query, doc_id)` | `Σ IDF × (tf × (k1+1)) / (tf + k1 × (1-b+b×dl/avgdl))` | Okapi BM25 with saturation and length normalization |

**BM25 Parameters:**
- `k1 = 1.5`: Controls term frequency saturation (higher = more weight to repeated terms)
- `b = 0.75`: Controls document length normalization (0 = no normalization, 1 = full normalization)

**Search Methods:**
- `search_bm25()`: Ranks all candidate docs by BM25 score
- `search_tfidf()`: Ranks by TF-IDF cosine similarity
- `boolean_search()`: AND/OR set operations on posting lists

**Persistence:** `save_index()` / `load_index()` use Python pickle for fast serialization.

### Build the Index

In [ ]:
from preprocessor import load_processed
processed_docs = load_processed('data/processed_articles.json')

index = InvertedIndex()
index.build_index(processed_docs)
index.save_index('data/inverted_index.pkl')

### 📘 Explanation — Building the Index

1. **Load processed documents** from the JSON file
2. **Create `InvertedIndex` instance** and call `build_index()`
3. The builder iterates through all documents, counting term frequencies and building posting lists
4. IDF values are precomputed for all terms
5. **`save_index()`** serializes the entire index to `data/inverted_index.pkl` using pickle

The output shows: documents indexed, vocabulary size, average document length, and total postings (sum of all posting list lengths).

### Inspect the Index

In [ ]:
stats = index.get_index_stats()
print(f"Documents indexed:   {stats['total_documents']}")
print(f"Vocabulary size:     {stats['vocabulary_size']}")
print(f"Avg doc length:      {stats['avg_document_length']}")
print(f"Total postings:      {stats['total_postings']}")
print(f"Data sources:        {stats['sources']}")

In [ ]:
# Example: posting list for a term
term = 'intellig'  # stemmed form of 'intelligence'
postings = index.get_posting_list(term)
print(f"Posting list for '{term}': {len(postings)} documents")
for doc_id, freq in list(postings.items())[:5]:
    doc = index.get_document(doc_id)
    print(f"  doc={doc_id} freq={freq} title='{doc['title'][:60]}...')")

print(f"\nIDF('{term}'): {index.get_idf(term):.4f}")

### 📘 Explanation — Index Inspection

**Index Statistics:**
- **Documents indexed**: Total number of documents in the collection
- **Vocabulary size**: Number of unique terms in the index
- **Avg doc length**: Average number of tokens per document (used for BM25 length normalization)
- **Total postings**: Sum of all posting list lengths — represents the total (term, document) pairs stored
- **Data sources**: List of news sources represented in the index

**Posting List Example:**
- Shows the posting list for the stemmed term "intellig" (from "intelligence")
- Each entry in the posting list contains: document ID, term frequency, and document title
- The IDF value shows how discriminative the term is — higher IDF means the term appears in fewer documents (more useful for ranking)

---
## 5. Search & Retrieval

The search engine:
- Preprocesses the query using the same pipeline as documents
- Retrieves and ranks documents using BM25, TF-IDF, or Boolean modes
- Generates relevant snippets from the best-matching sentences
- Highlights matched query words in titles and snippets

In [ ]:
"""
Search Engine Module for News IR System
=========================================
Handles query processing, search execution, and result formatting.
Supports keyword highlighting in titles and snippets.
"""

import re


class SearchEngine:
    """
    Search engine that processes queries and returns ranked results.
    Supports TF-IDF, BM25, and Boolean search modes.
    """

    def __init__(self, index: InvertedIndex, preprocessor: TextPreprocessor = None):
        self.index = index
        self.preprocessor = preprocessor or TextPreprocessor()

    def search(self, query, mode="bm25", top_k=10):
        """
        Execute a search query.

        Args:
            query: Raw query string
            mode: 'bm25', 'tfidf', or 'boolean'
            top_k: Number of results to return

        Returns:
            List of result dicts with doc info and scores
        """
        # Preprocess query and keep original words for snippet matching
        query_tokens, query_original_words = self.preprocessor.get_original_words_for_query(query)

        if not query_tokens:
            return []

        # Execute search based on mode
        if mode == "bm25":
            ranked = self.index.search_bm25(query_tokens, top_k)
        elif mode == "tfidf":
            ranked = self.index.search_tfidf(query_tokens, top_k)
        elif mode == "boolean":
            doc_ids = self.index.boolean_search(query_tokens, "AND")
            ranked = []
            for doc_id in doc_ids:
                score = self.index.bm25_score(query_tokens, doc_id)
                ranked.append((doc_id, score))
            ranked.sort(key=lambda x: x[1], reverse=True)
            ranked = ranked[:top_k]
        else:
            ranked = self.index.search_bm25(query_tokens, top_k)

        # Build the set of highlight words (original + stemmed for broader matching)
        highlight_words = set(w.lower() for w in query_original_words)
        highlight_stems = set(query_tokens)

        # Format results
        results = []
        for rank, (doc_id, score) in enumerate(ranked, 1):
            doc = self.index.get_document(doc_id)
            if doc:
                snippet = self._generate_snippet(doc_id, query_original_words, query_tokens)
                highlighted_title = self._highlight_text(doc["title"], highlight_words, highlight_stems)
                highlighted_snippet = self._highlight_text(snippet, highlight_words, highlight_stems)

                results.append({
                    "rank": rank,
                    "doc_id": doc_id,
                    "score": round(score, 4),
                    "title": doc["title"],
                    "title_highlighted": highlighted_title,
                    "snippet": snippet,
                    "snippet_highlighted": highlighted_snippet,
                    "source": doc["source"],
                    "url": doc["url"],
                    "published_date": doc["published_date"],
                })

        return results

    def _highlight_text(self, text, highlight_words, highlight_stems):
        """
        Highlight matching query words in text by wrapping them in <mark> tags.
        
        Matches on:
          1. Exact original query words (e.g. "intelligence" highlights "intelligence")
          2. Words whose stem matches a query stem (e.g. query "technology" stems to
             "technolog", which also matches "technologies", "technological", etc.)
        """
        if not text or (not highlight_words and not highlight_stems):
            return text

        stemmer = self.preprocessor.stemmer

        # Split text into tokens while preserving separators (spaces, punctuation)
        # This regex splits on word boundaries, keeping all parts
        parts = re.split(r'(\b\w+\b)', text)

        result = []
        for part in parts:
            if not part:
                continue
            # Check if this part is a word
            if re.match(r'^\w+$', part):
                word_lower = part.lower()
                matched = False

                # Check 1: exact match with original query words
                if word_lower in highlight_words:
                    matched = True

                # Check 2: stem match — stem the word and see if it matches any query stem
                if not matched:
                    try:
                        word_stem = stemmer.stem(word_lower)
                        if word_stem in highlight_stems:
                            matched = True
                    except Exception:
                        pass

                if matched:
                    result.append(f'<mark>{part}</mark>')
                else:
                    result.append(part)
            else:
                result.append(part)

        return ''.join(result)

    def _generate_snippet(self, doc_id, query_original_words, query_stemmed_tokens, max_length=300):
        """
        Generate a relevant text snippet for a document.
        
        Uses ORIGINAL query words (not stemmed) to find matching sentences
        in the original document content, so snippets are actually relevant.
        """
        doc = self.index.get_document(doc_id)
        if not doc:
            return ""

        content = doc.get("original_content", "")
        if not content:
            return ""

        # Split into sentences properly
        try:
            from nltk.tokenize import sent_tokenize
            sentences = sent_tokenize(content)
        except Exception:
            sentences = re.split(r'[.!?]+\s+', content)

        if not sentences:
            return content[:max_length] + "..."

        # Score each sentence by how many ORIGINAL query words it contains
        scored_sentences = []
        for sentence in sentences:
            sentence_lower = sentence.lower()
            score = 0
            for word in query_original_words:
                if word.lower() in sentence_lower:
                    score += 2
            for stem in query_stemmed_tokens:
                if len(stem) >= 3 and stem in sentence_lower:
                    score += 1
            scored_sentences.append((score, sentence))

        scored_sentences.sort(key=lambda x: x[0], reverse=True)

        if scored_sentences[0][0] > 0:
            snippet_parts = []
            total_len = 0
            for score, sent in scored_sentences:
                if score == 0:
                    break
                if total_len + len(sent) > max_length:
                    if not snippet_parts:
                        snippet_parts.append(sent[:max_length])
                    break
                snippet_parts.append(sent)
                total_len += len(sent)
            snippet = " ".join(snippet_parts)
        else:
            snippet = content[:max_length]

        if len(snippet) < len(content):
            snippet += "..."

        return snippet

    def display_results(self, results, query):
        """Display search results in a formatted way."""
        print(f"\n{'=' * 60}")
        print(f"  Search Results for: \"{query}\"")
        print(f"  Found {len(results)} results")
        print(f"{'=' * 60}")

        if not results:
            print("  No results found.")
            return

        for r in results:
            print(f"\n  [{r['rank']}] {r['title']}")
            print(f"      Score:  {r['score']}")
            print(f"      Source: {r['source']}")
            print(f"      URL:    {r['url']}")
            print(f"      {r['snippet']}")
            print(f"      {'-' * 50}")


### 📘 Explanation — Search Engine Module

**Class: `SearchEngine`**

**`search(query, mode, top_k)`** — Main search method:
1. **Query preprocessing**: Applies the SAME preprocessing pipeline as documents (normalize → tokenize → remove stop words → lemmatize → stem). This ensures query terms match indexed terms.
2. **Also keeps original query words** for snippet matching (via `get_original_words_for_query()`)
3. **Retrieval**: Calls the appropriate index search method (BM25/TF-IDF/Boolean)
4. **Snippet generation**: For each result, finds the most relevant sentences
5. **Highlighting**: Wraps matching words in `<mark>` tags

**`_highlight_text(text, highlight_words, highlight_stems)`:**
- Splits text into tokens while preserving punctuation/spacing
- Two-pass matching:
  1. **Exact match**: Checks if the word matches an original query word
  2. **Stem match**: Stems the word and checks against query stems (catches "technologies" for query "technology")
- Wraps matched words in `<mark>` HTML tags

**`_generate_snippet(doc_id, query_words, query_stems)`:**
- Splits original document content into sentences (using NLTK `sent_tokenize`)
- Scores each sentence by counting query word/stem matches
- Returns the highest-scoring sentences concatenated (up to 300 chars)
- Falls back to first 300 chars if no sentence matches

**`display_results()`:** Formatted console output with rank, title, score, source, URL, and snippet.

### Run Search Queries

In [ ]:
index = InvertedIndex()
index.load_index('data/inverted_index.pkl')
preprocessor = TextPreprocessor()
engine = SearchEngine(index, preprocessor)

In [ ]:
# Search: Artificial Intelligence
results = engine.search('artificial intelligence', mode='bm25', top_k=5)
engine.display_results(results, 'artificial intelligence')

In [ ]:
# Search: Climate Change
results = engine.search('climate change environment', mode='bm25', top_k=5)
engine.display_results(results, 'climate change environment')

In [ ]:
# Search: War & Military
results = engine.search('war conflict military', mode='bm25', top_k=5)
engine.display_results(results, 'war conflict military')

### 📘 Explanation — Running Search Queries

1. **Load the saved index** from `data/inverted_index.pkl`
2. **Create SearchEngine** with the index and a fresh preprocessor
3. **Execute queries** with BM25 ranking (default, top 5 results)

Each result shows:
- **Rank**: Position in the ranked list
- **Title**: Article title with query word highlighting
- **Score**: BM25 relevance score (higher = more relevant)
- **Source**: Which news outlet published the article
- **URL**: Link to the original article
- **Snippet**: Most relevant sentence(s) from the article content

### Compare BM25 vs TF-IDF

In [ ]:
query = 'technology artificial intelligence'
print('=== BM25 Results ===')
bm25_results = engine.search(query, mode='bm25', top_k=5)
for r in bm25_results:
    print(f"  [{r['rank']}] Score={r['score']:.4f} | {r['title'][:70]}")

print('\n=== TF-IDF Results ===')
tfidf_results = engine.search(query, mode='tfidf', top_k=5)
for r in tfidf_results:
    print(f"  [{r['rank']}] Score={r['score']:.4f} | {r['title'][:70]}")

### 📘 Explanation — BM25 vs TF-IDF Comparison

This cell runs the same query with both ranking algorithms to compare:

**BM25 (Okapi BM25):**
- Uses term frequency saturation (diminishing returns for repeated terms)
- Applies document length normalization (shorter docs don't get unfairly penalized)
- Generally produces better rankings for natural language queries

**TF-IDF:**
- Log-normalized term frequency × Inverse Document Frequency
- No explicit length normalization or saturation
- Simpler but can over-reward long documents with many term occurrences

**What to observe:**
- BM25 and TF-IDF may rank the same documents differently
- BM25 typically produces more relevant top results
- Score scales differ between the two methods (not directly comparable)

---
## 6. Evaluation & Quality Metrics

We evaluate using **keyword-based relevance judgments** — a document is relevant if its title/content contains topic-specific keywords.

**Metrics:**
- **Precision@K**: fraction of top-K results that are relevant
- **Recall@K**: fraction of all relevant docs found in top-K
- **F1@K**: harmonic mean of P and R
- **MAP**: Mean Average Precision across all queries
- **NDCG@K**: Normalized Discounted Cumulative Gain

In [ ]:
"""
Evaluation Module for News IR System
======================================
Evaluates IR system quality using Precision, Recall, F1 Score,
Mean Average Precision (MAP), and Normalized Discounted Cumulative Gain (NDCG).

Uses keyword-based relevance judgments: a document is relevant to a query
if its TITLE or CONTENT contains any of the specified relevance keywords.
"""

import json
import os
import re
import math
from collections import defaultdict


class IREvaluator:
    """
    Evaluates IR system performance using standard metrics.
    Uses keyword-based relevance judgments for test queries.
    """

    def __init__(self, search_engine):
        self.search_engine = search_engine
        self.results = {}

    def create_test_queries(self, index):
        """
        Create test queries with ground-truth relevance judgments.
        
        Relevance is determined by checking if a document's TITLE or CONTENT
        contains specific keywords related to the query topic. This provides
        honest, content-based relevance judgments rather than circular 
        pseudo-relevance from the ranking algorithm itself.
        """
        # Each test query has:
        #   - query: the search query string
        #   - description: what the query is about
        #   - relevance_keywords: words that MUST appear in a relevant doc's
        #     title or content (at least one keyword must match)
        test_query_definitions = [
            {
                "query": "technology artificial intelligence",
                "description": "AI and tech news",
                "relevance_keywords": ["artificial intelligence", "ai ", " ai,", "machine learning",
                                       "neural network", "deep learning", "chatbot", "openai",
                                       "google ai", "robot"],
            },
            {
                "query": "climate change environment",
                "description": "Environmental and climate news",
                "relevance_keywords": ["climate", "global warming", "carbon", "emission",
                                       "greenhouse", "environment", "renewable energy",
                                       "fossil fuel", "temperature rise", "sea level"],
            },
            {
                "query": "economic growth market",
                "description": "Economy and financial markets",
                "relevance_keywords": ["economy", "economic", "gdp", "stock market", "inflation",
                                       "interest rate", "trade", "recession", "financial",
                                       "growth rate", "investor"],
            },
            {
                "query": "government policy election",
                "description": "Political news and elections",
                "relevance_keywords": ["election", "government", "president", "minister",
                                       "parliament", "vote", "political", "policy",
                                       "legislation", "democrat", "republican", "party"],
            },
            {
                "query": "health medical research",
                "description": "Health and medical news",
                "relevance_keywords": ["health", "medical", "doctor", "hospital", "patient",
                                       "disease", "treatment", "drug", "vaccine", "clinical",
                                       "symptom", "diagnosis", "cancer"],
            },
            {
                "query": "war conflict military",
                "description": "Military and conflict news",
                "relevance_keywords": ["war", "military", "army", "soldier", "conflict",
                                       "attack", "weapon", "troops", "invasion", "ceasefire",
                                       "defense", "nato", "combat"],
            },
            {
                "query": "sports football championship",
                "description": "Sports news",
                "relevance_keywords": ["sport", "football", "soccer", "championship", "league",
                                       "match", "team", "player", "score", "tournament",
                                       "olympic", "coach", "game"],
            },
            {
                "query": "education university student",
                "description": "Education news",
                "relevance_keywords": ["education", "university", "student", "school", "teacher",
                                       "college", "academic", "degree", "tuition", "campus",
                                       "learning", "curriculum"],
            },
            {
                "query": "energy oil gas renewable",
                "description": "Energy sector news",
                "relevance_keywords": ["energy", "oil", "gas", "renewable", "solar", "wind power",
                                       "nuclear", "electricity", "power plant", "fuel",
                                       "petroleum", "opec"],
            },
            {
                "query": "security cyber data privacy",
                "description": "Cybersecurity and privacy news",
                "relevance_keywords": ["cyber", "security", "hack", "data breach", "privacy",
                                       "encryption", "malware", "ransomware", "phishing",
                                       "firewall", "surveillance", "data protection"],
            },
        ]

        test_queries = []

        for tq_def in test_query_definitions:
            # Find ALL documents in the corpus that match the relevance keywords
            relevant_docs = set()
            for doc_id, doc_meta in index.documents.items():
                title = doc_meta.get("title", "").lower()
                content = doc_meta.get("original_content", "").lower()
                combined = title + " " + content

                for keyword in tq_def["relevance_keywords"]:
                    if keyword.lower() in combined:
                        relevant_docs.add(doc_id)
                        break  # One keyword match is enough

            # Only include queries that have at least 2 relevant documents
            if len(relevant_docs) >= 2:
                test_queries.append({
                    "query": tq_def["query"],
                    "description": tq_def["description"],
                    "relevant_docs": list(relevant_docs),
                    "relevance_keywords": tq_def["relevance_keywords"],
                })

        print(f"[+] Created {len(test_queries)} test queries with keyword-based relevance judgments")
        for tq in test_queries:
            print(f"    - \"{tq['query']}\": {len(tq['relevant_docs'])} relevant docs")
        return test_queries

    def precision_at_k(self, retrieved_ids, relevant_ids, k):
        """Precision@K: fraction of top-K results that are relevant."""
        retrieved_k = retrieved_ids[:k]
        relevant_count = sum(1 for doc_id in retrieved_k if doc_id in relevant_ids)
        return relevant_count / k if k > 0 else 0

    def recall_at_k(self, retrieved_ids, relevant_ids, k):
        """Recall@K: fraction of relevant documents found in top-K."""
        retrieved_k = set(retrieved_ids[:k])
        relevant_found = len(retrieved_k & relevant_ids)
        return relevant_found / len(relevant_ids) if relevant_ids else 0

    def f1_score(self, precision, recall):
        """F1 Score: harmonic mean of precision and recall."""
        if precision + recall == 0:
            return 0
        return 2 * (precision * recall) / (precision + recall)

    def average_precision(self, retrieved_ids, relevant_ids):
        """Average Precision for a single query."""
        hits = 0
        sum_precision = 0

        for i, doc_id in enumerate(retrieved_ids):
            if doc_id in relevant_ids:
                hits += 1
                sum_precision += hits / (i + 1)

        return sum_precision / len(relevant_ids) if relevant_ids else 0

    def dcg_at_k(self, retrieved_ids, relevant_ids, k):
        """Discounted Cumulative Gain at K."""
        dcg = 0
        for i, doc_id in enumerate(retrieved_ids[:k]):
            rel = 1 if doc_id in relevant_ids else 0
            dcg += rel / math.log2(i + 2)  # i+2 because log2(1) = 0
        return dcg

    def ndcg_at_k(self, retrieved_ids, relevant_ids, k):
        """Normalized DCG at K."""
        dcg = self.dcg_at_k(retrieved_ids, relevant_ids, k)
        # Ideal DCG: all relevant docs at top
        ideal_retrieved = list(relevant_ids)[:k]
        ideal_dcg = self.dcg_at_k(ideal_retrieved, relevant_ids, k)
        return dcg / ideal_dcg if ideal_dcg > 0 else 0

    def evaluate(self, test_queries, k_values=[5, 10]):
        """
        Run full evaluation on test queries.

        Args:
            test_queries: List of {query, relevant_docs} dicts
            k_values: List of K values for Precision@K, Recall@K

        Returns:
            Evaluation results dict
        """
        print("\n" + "=" * 60)
        print("  IR SYSTEM EVALUATION")
        print("=" * 60)

        all_results = {
            "per_query": [],
            "aggregate": {},
        }

        # Metrics accumulators
        metrics = defaultdict(list)

        for i, tq in enumerate(test_queries):
            query = tq["query"]
            relevant_ids = set(tq["relevant_docs"])

            # Search with BM25
            results = self.search_engine.search(query, mode="bm25", top_k=max(k_values))
            retrieved_ids = [r["doc_id"] for r in results]

            query_metrics = {
                "query": query,
                "description": tq.get("description", ""),
                "num_relevant": len(relevant_ids),
                "num_retrieved": len(retrieved_ids),
            }

            for k in k_values:
                p_k = self.precision_at_k(retrieved_ids, relevant_ids, k)
                r_k = self.recall_at_k(retrieved_ids, relevant_ids, k)
                f1 = self.f1_score(p_k, r_k)
                ndcg = self.ndcg_at_k(retrieved_ids, relevant_ids, k)

                query_metrics[f"P@{k}"] = round(p_k, 4)
                query_metrics[f"R@{k}"] = round(r_k, 4)
                query_metrics[f"F1@{k}"] = round(f1, 4)
                query_metrics[f"NDCG@{k}"] = round(ndcg, 4)

                metrics[f"P@{k}"].append(p_k)
                metrics[f"R@{k}"].append(r_k)
                metrics[f"F1@{k}"].append(f1)
                metrics[f"NDCG@{k}"].append(ndcg)

            ap = self.average_precision(retrieved_ids, relevant_ids)
            query_metrics["AP"] = round(ap, 4)
            metrics["AP"].append(ap)

            all_results["per_query"].append(query_metrics)

            print(f"\n  Query {i+1}: \"{query}\"")
            print(f"    Relevant: {len(relevant_ids)}, Retrieved: {len(retrieved_ids)}")
            
            # Show which retrieved docs are relevant/not relevant
            for j, r_id in enumerate(retrieved_ids[:5]):
                is_rel = "RELEVANT" if r_id in relevant_ids else "NOT RELEVANT"
                doc = self.search_engine.index.get_document(r_id)
                title = doc["title"][:50] if doc else "?"
                print(f"      #{j+1} [{is_rel}] {title}...")
            
            for k in k_values:
                print(f"    P@{k}={query_metrics[f'P@{k}']:.3f}  "
                      f"R@{k}={query_metrics[f'R@{k}']:.3f}  "
                      f"F1@{k}={query_metrics[f'F1@{k}']:.3f}  "
                      f"NDCG@{k}={query_metrics[f'NDCG@{k}']:.3f}")

        # Compute aggregate metrics
        print(f"\n{'-' * 60}")
        print("  AGGREGATE METRICS (averaged over all queries)")
        print(f"{'-' * 60}")

        for metric_name, values in sorted(metrics.items()):
            avg_val = sum(values) / len(values) if values else 0
            all_results["aggregate"][f"Mean_{metric_name}"] = round(avg_val, 4)
            print(f"    Mean {metric_name}: {avg_val:.4f}")

        # MAP
        map_score = sum(metrics["AP"]) / len(metrics["AP"]) if metrics["AP"] else 0
        all_results["aggregate"]["MAP"] = round(map_score, 4)
        print(f"    MAP:      {map_score:.4f}")
        print("=" * 60)

        self.results = all_results
        return all_results

    def save_evaluation(self, filepath="data/evaluation_results.json"):
        """Save evaluation results to file."""
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(self.results, f, indent=2)
        print(f"[+] Evaluation results saved to {filepath}")

    def generate_report(self):
        """Generate a formatted evaluation report string."""
        if not self.results:
            return "No evaluation results available."

        report = []
        report.append("=" * 60)
        report.append("  IR SYSTEM EVALUATION REPORT")
        report.append("=" * 60)
        report.append(f"\n  Number of test queries: {len(self.results['per_query'])}")
        report.append("\n  Aggregate Metrics:")
        for metric, value in self.results["aggregate"].items():
            report.append(f"    {metric}: {value:.4f}")

        report.append(f"\n{'-' * 60}")
        report.append("  Per-Query Results:")
        for pq in self.results["per_query"]:
            report.append(f"\n  Query: \"{pq['query']}\"")
            report.append(f"    Description: {pq.get('description', '')}")
            for key, val in pq.items():
                if key not in ["query", "description"]:
                    report.append(f"    {key}: {val}")

        return "\n".join(report)


### 📘 Explanation — Evaluation Module

**Class: `IREvaluator`**

**Relevance Judgment Strategy:**
Instead of manual human relevance judgments (expensive), we use **keyword-based relevance**:
- Each test query has a list of topic-specific keywords (e.g., "artificial intelligence" query uses keywords: "ai", "machine learning", "neural network", "deep learning", etc.)
- A document is marked **relevant** if its title or content contains ANY of these keywords
- This provides honest, content-based judgments independent of the ranking algorithm

**10 Test Query Topics:** AI/Tech, Climate, Economy, Politics, Health, Military, Sports, Education, Energy, Cybersecurity

**Evaluation Metrics:**

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **Precision@K** | `relevant_in_top_K / K` | What fraction of returned results are actually relevant? |
| **Recall@K** | `relevant_found / total_relevant` | What fraction of ALL relevant docs did we find? |
| **F1@K** | `2 × P × R / (P + R)` | Harmonic mean — balances precision and recall |
| **AP** | `Σ (P@i × rel_i) / total_relevant` | Average of precision values at each relevant document's rank |
| **MAP** | `mean(AP) across all queries` | Single-number summary of overall system quality |
| **DCG@K** | `Σ rel_i / log2(i+1)` | Cumulative gain with position discount (early ranks matter more) |
| **NDCG@K** | `DCG / ideal_DCG` | Normalized to [0,1] — 1.0 means perfect ranking |

**`create_test_queries(index)`:**
- Scans ALL documents for keyword matches to build ground truth
- Only includes queries with ≥ 2 relevant documents

**`evaluate(test_queries, k_values)`:**
- Runs each query through the search engine (BM25 mode)
- Computes all metrics at each K value
- Shows per-query breakdown with relevant/not-relevant labels
- Computes aggregate (averaged) metrics across all queries

### Run Evaluation

In [ ]:
evaluator = IREvaluator(engine)
test_queries = evaluator.create_test_queries(index)
eval_results = evaluator.evaluate(test_queries, k_values=[5, 10])
evaluator.save_evaluation()

### 📘 Explanation — Running Evaluation

1. **Create evaluator** with our search engine
2. **`create_test_queries(index)`**: Scans the corpus to find which documents match each topic's keywords, building ground-truth relevance sets
3. **`evaluate(test_queries, k_values=[5, 10])`**: Runs all test queries, computes metrics at K=5 and K=10
4. **`save_evaluation()`**: Persists results to `data/evaluation_results.json`

The output shows for each query:
- Number of relevant documents found in the corpus
- Top-5 retrieved documents labeled as RELEVANT or NOT RELEVANT
- All metrics at K=5 and K=10

### Evaluation Summary Table

In [ ]:
import pandas as pd

# Per-query results
df_eval = pd.DataFrame(eval_results['per_query'])
cols = ['query', 'num_relevant', 'P@5', 'R@5', 'F1@5', 'NDCG@5', 'AP']
df_eval[cols]

In [ ]:
# Aggregate metrics
print('Aggregate Evaluation Metrics:')
print('=' * 40)
for metric, value in sorted(eval_results['aggregate'].items()):
    print(f'  {metric:20s}: {value:.4f}')

### 📘 Explanation — Evaluation Results Table

**Per-Query Table:**
Shows each test query with its metrics — useful for identifying which topics the system handles well vs. poorly.

**Aggregate Metrics:**
- **Mean P@5**: Average precision across all queries at top-5 — "How accurate are the top results?"
- **Mean R@5**: Average recall at top-5 — "How complete are the top results?"
- **Mean F1@5**: Balanced accuracy measure
- **MAP**: Single-number quality summary — the most widely used IR metric
- **Mean NDCG@5**: Ranking quality — penalizes relevant documents appearing at lower positions

**Interpreting Results:**
- MAP > 0.3 is reasonable for a basic IR system
- P@5 > 0.6 means most top-5 results are relevant
- Low recall is expected when K is small relative to total relevant documents

---
## 7. Summary & Conclusions

### System Architecture
| Component | Description |
|-----------|-------------|
| Data Collection | RSS feeds + BeautifulSoup web scraping from 6 news sources |
| Preprocessing | Normalization → Tokenization → Stop Words → Lemmatization → Stemming |
| Indexing | Inverted Index with TF-IDF and BM25 scoring, pickle persistence |
| Search | Ranked retrieval with snippet generation and keyword highlighting |
| Evaluation | P@K, R@K, F1, MAP, NDCG with keyword-based relevance judgments |

### Key Design Decisions
1. **Title Boosting (3×)**: Title tokens are repeated 3 times during indexing to give them higher weight — titles are more informative than body text
2. **Lemmatization + Stemming**: Both are applied (lemmatization first, then stemming) for maximum recall
3. **BM25 over TF-IDF**: BM25's term saturation and length normalization produce better rankings
4. **Keyword-based Evaluation**: Provides honest relevance judgments without expensive human annotation

### Technologies Used
- **Python** — Core language
- **NLTK** — Tokenization, stemming, lemmatization, stop words
- **BeautifulSoup** — Web scraping and HTML parsing
- **Feedparser** — RSS feed parsing
- **Flask** — Web interface for interactive search

---
## 8. Web Interface (Flask)

The project includes a Flask web app with a clean editorial search UI.

```bash
# To run the web interface:
python app.py
# Open http://127.0.0.1:5000
```

In [ ]:
# Flask Web Application Code (for reference)
# Run separately with: python app.py

"""
Flask Web Application for News IR System
==========================================
Beautiful web-based search interface for the News IR System.
"""

import os
import json
from flask import Flask, render_template, request, jsonify

from preprocessor import TextPreprocessor
from indexer import InvertedIndex
from search_engine import SearchEngine

app = Flask(__name__, template_folder="templates", static_folder="static")

# Global objects
index = InvertedIndex()
preprocessor = TextPreprocessor()
engine = None


def init_engine():
    """Initialize the search engine with saved index."""
    global engine
    if index.load_index("data/inverted_index.pkl"):
        engine = SearchEngine(index, preprocessor)
        print("[+] Search engine ready!")
    else:
        print("[!] No index found. Run 'python news_ir_system.py --build' first.")


@app.route("/")
def home():
    """Render the search home page."""
    stats = index.get_index_stats() if index.total_docs > 0 else {}
    
    # Load evaluation results if available
    eval_results = {}
    eval_path = "data/evaluation_results.json"
    if os.path.exists(eval_path):
        with open(eval_path, "r") as f:
            eval_results = json.load(f)
    
    return render_template("index.html", stats=stats, eval_results=eval_results)


@app.route("/search")
def search():
    """Execute a search query and return results."""
    query = request.args.get("q", "").strip()
    mode = request.args.get("mode", "bm25")
    top_k = int(request.args.get("top_k", 10))

    if not query or not engine:
        return render_template("index.html", query=query, results=[], stats=index.get_index_stats(), eval_results={})

    results = engine.search(query, mode=mode, top_k=top_k)
    stats = index.get_index_stats()
    
    return render_template(
        "index.html",
        query=query,
        results=results,
        mode=mode,
        stats=stats,
        eval_results={},
    )


@app.route("/api/search")
def api_search():
    """REST API endpoint for search."""
    query = request.args.get("q", "").strip()
    mode = request.args.get("mode", "bm25")
    top_k = int(request.args.get("top_k", 10))

    if not query or not engine:
        return jsonify({"error": "No query provided or engine not initialized", "results": []})

    results = engine.search(query, mode=mode, top_k=top_k)
    return jsonify({"query": query, "mode": mode, "count": len(results), "results": results})


@app.route("/stats")
def stats_page():
    """Show index and evaluation statistics."""
    stats = index.get_index_stats() if index.total_docs > 0 else {}
    eval_results = {}
    eval_path = "data/evaluation_results.json"
    if os.path.exists(eval_path):
        with open(eval_path, "r") as f:
            eval_results = json.load(f)
    return render_template("index.html", stats=stats, eval_results=eval_results, show_stats=True)


if __name__ == "__main__":
    init_engine()
    print("\n[*] Starting web server at http://127.0.0.1:5000")
    app.run(debug=False, host="127.0.0.1", port=5000)


### 📘 Explanation — Flask Web Application

The web interface provides a browser-based search experience:

**Routes:**
- `GET /` — Home page with search bar and system statistics
- `GET /search?q=...&mode=bm25` — Execute search and display results
- `GET /api/search?q=...` — REST API endpoint returning JSON results
- `GET /stats` — Statistics dashboard

**Features:**
- **Search mode selection**: Toggle between BM25, TF-IDF, and Boolean AND via pill buttons
- **Query highlighting**: Matched words in titles and snippets are wrapped in `<mark>` tags
- **Source tags**: Color-coded badges showing which news outlet each result comes from
- **Statistics dashboard**: Document count, vocabulary size, avg doc length, postings count
- **Evaluation metrics display**: MAP, P@5, R@5, F1@5 with per-query breakdown table

**Design:** Clean, editorial-style UI inspired by news websites, using Source Serif 4 and Inter fonts with a warm color palette.

**Note:** This cell is for reference only — the Flask app must be run as a standalone script with `python app.py`, not from within the notebook.